# Fetch Historical Weather Data for Austria

This notebook fetches historical weather data from OpenWeather API for Austria using its ENTSO-E bidding zone coordinates.

In [1]:
import json
import pandas as pd
import requests
from datetime import datetime, timedelta
import time
import os
from pathlib import Path

In [17]:
# Load the bidding zone centroids from CSV
centroids_df = pd.read_csv('Valid_Bidding_Zone_Centroids.csv')
print("Loaded coordinates for the following zones:")
print(centroids_df[['zone', 'lat', 'lon']])

Loaded coordinates for the following zones:
      zone        lat        lon
0       AT  47.585657  14.130673
1       BE  50.637462   4.639961
2       BG  42.768523  25.217513
3       CZ  49.732448  15.314103
4       HU  47.163244  19.396759
5   IT_SUD  40.937624  16.188191
6       LV  56.851819  24.908622
7     NO_1  60.915127  10.834545
8     NO_2  59.099094   7.379213
9     NO_3  63.062412   8.812196
10    NO_4  68.782782  19.747524
11    NO_5  60.746237   6.947801
12      PL  52.124772  19.394322
13      RO  45.853464  24.972942
14      SK  48.704935  19.480383
15      SI  46.116119  14.808104
16      CH  46.798957   8.215354
17      RS  44.222148  20.789252


In [ ]:
BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'

def fetch_historical_weather(lat, lon, start_date, end_date, zone='AT', max_retries=3):
    hourly_vars = [
        'temperature_2m',
        'relativehumidity_2m',
        'cloudcover',
        'wind_speed_10m',
        'winddirection_10m',
        'pressure_msl',
        'shortwave_radiation'
    ]
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date.strftime('%Y-%m-%d'),
        'end_date':   end_date.strftime('%Y-%m-%d'),
        'hourly':     ','.join(hourly_vars),
        'timezone':   'UTC'
    }

    for attempt in range(1, max_retries+1):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            break
        except requests.exceptions.HTTPError as e:
            if resp.status_code == 504 and attempt < max_retries:
                wait = 5 * attempt
                print(f"  → 504 on {start_date.date()}–{end_date.date()}, retry in {wait}s…")
                time.sleep(wait)
                continue
            else:
                raise

    out = []
    times = data['hourly']['time']
    for i, tstr in enumerate(times):
        ts = datetime.fromisoformat(tstr)
        out.append({
            'timestamp':        ts,
            'temp':             data['hourly']['temperature_2m'][i],
            'humidity':         data['hourly']['relativehumidity_2m'][i],
            'cloudcover':       data['hourly']['cloudcover'][i],
            'wind_speed':       data['hourly']['wind_speed_10m'][i],
            'wind_direction':   data['hourly']['winddirection_10m'][i],
            'pressure':         data['hourly']['pressure_msl'][i],
            'shortwave_radiation': data['hourly']['shortwave_radiation'][i],
            'zone':             zone
        })
    return out

In [ ]:
# Set up time range
start = datetime(2015, 1, 1)
end   = datetime(2019, 12, 31)

# Process each zone
for _, zone in centroids_df.iterrows():
    print(f"\nProcessing zone: {zone['zone']}")
    print(f"Coordinates: lat={zone['lat']}, lon={zone['lon']}")
    
    current_start = start
    zone_data = []

    while current_start <= end:
        # month end
        next_month = (current_start.replace(day=1) + timedelta(days=32)).replace(day=1) - timedelta(days=1)
        slice_end = min(next_month, end)
        
        print(f"Fetching {current_start.date()} → {slice_end.date()} …")
        chunk = fetch_historical_weather(zone['lat'], zone['lon'], current_start, slice_end, zone=zone['zone'])
        zone_data.extend(chunk)
        
        # advance to next month
        current_start = slice_end + timedelta(days=1)

    # Convert to DataFrame and save for this zone
    df = pd.DataFrame(zone_data)
    
    # Make sure folder exists and save
    os.makedirs('weather_data', exist_ok=True)
    outfile = f'weather_data/{zone["zone"]}_hourly_2015-2019.csv'
    df.to_csv(outfile, index=False)
    print(f"Saved {len(df)} records to {outfile}")


Processing zone: AT
Coordinates: lat=47.58565692052221, lon=14.13067288731315

Processing zone: BE
Coordinates: lat=50.63746163533436, lon=4.639960507668175

Processing zone: BG
Coordinates: lat=42.76852261435961, lon=25.217512511716745

Processing zone: CZ
Coordinates: lat=49.73244820549173, lon=15.31410310325546

Processing zone: HU
Coordinates: lat=47.16324368659333, lon=19.396759267358192

Processing zone: IT_SUD
Coordinates: lat=40.93762358269949, lon=16.188191251086153

Processing zone: LV
Coordinates: lat=56.85181947960624, lon=24.908622427569245

Processing zone: NO_1
Coordinates: lat=60.91512697794608, lon=10.83454502741289

Processing zone: NO_2
Coordinates: lat=59.099093824760565, lon=7.379212609314895

Processing zone: NO_3
Coordinates: lat=63.06241203608625, lon=8.812195670009919

Processing zone: NO_4
Coordinates: lat=68.78278187673045, lon=19.747523767188188

Processing zone: NO_5
Coordinates: lat=60.74623714657913, lon=6.947801095455383

Processing zone: PL
Coordinates

In [20]:
# Print summary of files created
print("\nSummary of created files:")
for file in os.listdir('weather_data'):
    if file.endswith('2015-2019.csv'):
        df = pd.read_csv(os.path.join('weather_data', file))
        print(f"{file}: {len(df)} records")



Summary of created files:
RS_hourly_2015-2019.csv: 43824 records
